# 第3日 ノートブック03：検出結果を読み解く

# Day 3 Notebook 03: Understanding Detection Results

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. confidence のしきい値を変えて結果を比較できる | 1. Compare results across confidence thresholds |
| 2. 検出結果の座標を読み、意味を説明できる | 2. Read detection coordinates and explain them |
| 3. 偽陽性と偽陰性を、実際の画像で指摘できる | 3. Point out false positives and false negatives |
| 4. 画像サイズを下げると検出できなくなる理由を説明できる | 4. Explain why shrinking an image loses detections |
| 5. 実験結果を表にまとめて報告できる | 5. Record experiments in a table and report them |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 110 分 / 110 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第3日ノートブック01・02 を終えていること。 / You have finished Day 3 Notebooks 01 and 02. |
| 必要なライブラリ / Required libraries | torch, ultralytics, opencv-python, numpy, matplotlib, pandas |
| 使用カーネル / Jupyter kernel | aivision |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第3日 / Day 3](../../COURSE_SYLLABUS.md#day3) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| しきい値ごとの検出数の表 | A table of detection counts per threshold |
| 失敗例の記録（1 件以上） | At least one recorded failure case |

## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第3日 / Day 3 |
| この日の順番 / Position today | 3 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 110 分 / 110 minutes |
| 前提 / Prerequisites | `Day03_02_YOLO_Inference.ipynb` |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第3日 / Day 3](../../COURSE_SYLLABUS.md#day3) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.


---

## Part 0　準備 / Preparation

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("cv2", "opencv-python"),
    ("torch", "torch"),
    ("ultralytics", "ultralytics"),
    ("pandas", "pandas"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(14, 5)):
    """画像を並べて表示する / display images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def detections_to_frame(result):
    """検出結果を pandas の表にする / convert detections into a DataFrame.

    ``result.boxes.data`` の 6 列は
    (x1, y1, x2, y2, confidence, class_id) です。
    """
    columns = ["class_id", "name", "confidence", "x1", "y1", "x2", "y2"]
    boxes = result.boxes
    if len(boxes) == 0:
        return pd.DataFrame(columns=columns)

    data = boxes.data.cpu().numpy()
    return pd.DataFrame({
        "class_id": data[:, 5].astype(int),
        "name": [result.names[int(class_id)] for class_id in data[:, 5]],
        "confidence": data[:, 4].round(4),
        "x1": data[:, 0].round(2),
        "y1": data[:, 1].round(2),
        "x2": data[:, 2].round(2),
        "y2": data[:, 3].round(2),
    })

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import shutil
from pathlib import Path

MODELS_DIR = ASSETS / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def load_pretrained_yolo(model_name="yolo26n.pt"):
    """学習済みモデルを読み込む。初回だけダウンロードする。
    Load a pretrained model, downloading it only on the first run."""
    from ultralytics import YOLO

    weights_path = MODELS_DIR / model_name
    if not weights_path.is_file():
        print(f"初回のみダウンロードします / downloading {model_name} (first run only)...")
        print("  ネットワークに接続できない場合は、次を確認してください:")
        print("  - 社内プロキシの設定 / corporate proxy settings")
        print("  - もう一度このセルを実行する / run this cell again")
        model = YOLO(model_name)          # ダウンロードはカレントフォルダに行われる
        downloaded = Path(model_name)
        if downloaded.is_file():
            shutil.move(str(downloaded), str(weights_path))
            print("保存しました / cached to:", weights_path)
        else:
            raise RuntimeError(
                f"モデルを取得できませんでした / could not obtain {model_name}. "
                "ネットワークを確認して再実行してください / check the network and retry."
            )
    return YOLO(str(weights_path))

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import ultralytics
model = load_pretrained_yolo("yolo26n.pt")
sample_image = Path(ultralytics.__file__).parent / "assets" / "bus.jpg"
print("準備完了 / ready. image:", sample_image.is_file())

---

## Part 1　confidence のしきい値 / The confidence threshold

### しきい値で残る数が変わる / the threshold changes how many are kept

![](../../assets/diagrams/teach_confidence_filter.png)

図：同じ 1 回の推論で得た 4 つの検出を、しきい値 0.25・0.50・0.75 で絞り込んだ結果（合成画像・例示用の値）。

Figure: the same four detections from one inference, filtered at 0.25, 0.50 and 0.75 (synthetic image, illustrative scores).

**気づいてほしいこと**

推論は 1 回だけで、しきい値を変えても**やり直していません**。変わるのは「残すか捨てるか」だけです。しきい値を上げると残る数は減り、同時に、本当の欠陥も落としやすくなります。

**What to notice**

Inference ran once; changing the threshold does **not** run it again. Only the keep/drop decision changes. Raising the threshold keeps fewer boxes — and drops real defects too.

**考えてみてください / Try to answer**

- しきい値を 0.90 にすると、この 4 つのうち何個が残りますか。そのとき本当の欠陥は残りますか。
- At 0.90, how many of the four survive — and is the real defect still among them?


### 実際の結果で数える / counted on the real results

![](../../assets/reference_outputs/ref_confidence_counts.png)

図：この研修で実際に保存した推論結果（52 枚）を使い、しきい値を変えたときにNG と判定される枚数を数えたもの。右は画像ごとの最大 confidence の分布です。参考出力（実際の結果）。

Figure: using the 52 images actually saved by this course, how many are called NG at each threshold. Right: the distribution of each image's highest confidence. Reference output — real results.

**気づいてほしいこと**

推論は conf=0.01 で 1 回だけ行い、しきい値は後から動かしています。正常品でも 0.01〜0.1 の検出が残るため、しきい値を 0 にすると全数 NG になります。「検出がある」と「欠陥がある」は別だということが、この形で見えます。

**What to notice**

Inference ran once at conf = 0.01; the threshold is applied afterwards. Some normal parts still have a weak detection near 0.01, so a threshold of 0 calls everything NG. "A detection exists" is not the same as "a defect exists".

**考えてみてください / Try to answer**

- この図で、しきい値を 0.50 から 0.75 に上げると NG は何枚減りますか。減った分は正しい削除だと言い切れますか。
- Raising the threshold from 0.50 to 0.75 removes how many NG calls? Can you claim those removals were all correct?


### 検出は「四角の一覧」を返す / detection returns a list of boxes

![](../../assets/diagrams/diagram_detection.png)

図：検出が返すもの（模式図）。下の実験で、実際の四角と件数が変わります。

Figure: what detection returns (schematic). The experiment below changes the real boxes and counts.

**気づいてほしいこと**

4 枚は同じ 1 枚から作られています。しきい値を変えると **残る四角の数** が変わりますが、**モデルが見た候補そのものは変わりません**。捨てているだけです。だから「件数が 0 = 異常なし」ではありません。

**What to notice**

All four come from one image. Changing the threshold changes **how many boxes remain**, but the candidates the model produced do not change — they are only discarded. So "zero boxes" does not mean "no defect".

**考えてみてください / Try to answer**

- しきい値を 0.9 にすると、何が失われますか。
- What is lost when the threshold goes to 0.9?


#### しきい値は「どこまでを検出とみなすか」
モデルは、たくさんの候補を出します。その中から **confidence が
しきい値以上のものだけを採用** します。

| しきい値 | 結果 |
|---|---|
| 0.90 | ごく確かな検出だけ。**見逃し（偽陰性）が増える** |
| 0.25 | 標準的。バランスが良い |
| 0.05 | ほぼ全部採用。**誤検出（偽陽性）が増える** |

しきい値は **モデルではなく人間が決める** 数値です。
「見逃したくない」のか「誤報を減らしたい」のかで、変えるべき値が変わります。

#### A threshold decides what counts as a detection
The model produces many candidates; only those with **confidence above the threshold**
are kept.

| Threshold | Effect |
|---|---|
| 0.90 | only very certain detections; **more misses (false negatives)** |
| 0.25 | a common default with a reasonable balance |
| 0.05 | almost everything kept; **more false alarms (false positives)** |

The threshold is **chosen by a human, not by the model**. Whether you fear missing a defect or fear
false alarms determines the right value.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
thresholds = [0.05, 0.10, 0.25, 0.40, 0.60, 0.80, 0.90]
rows = []

for threshold in thresholds:
    prediction = model.predict(source=str(sample_image), conf=threshold, verbose=False)[0]
    boxes = prediction.boxes
    rows.append({
        "threshold": threshold,
        "detections": len(boxes),
        "classes": ", ".join(sorted({prediction.names[int(c)] for c in boxes.cls.tolist()})) if len(boxes) else "-",
        "max_conf": round(float(boxes.conf.max().item()), 3) if len(boxes) else 0.0,
    })

import pandas as pd
threshold_table = pd.DataFrame(rows)
print(threshold_table.to_string(index=False))

### 実行結果の例 / Expected output

次のような結果になります（数値は環境によって多少変わります）。

```text
threshold  detections                                     classes  max_conf
      0.05           9 bus, handbag, person, skateboard, stop sign     0.881
      0.10           6                      bus, person, stop sign     0.881
      0.25           5                                 bus, person     0.881
      0.40           5                                 bus, person     0.881
      0.60           5                                 bus, person     0.881
      0.80           4                                 bus, person     0.881
      0.90           0                                           -     0.000
```

You should see something like the following (exact numbers vary slightly between machines).

```text
threshold  detections  classes                                     max_conf
     0.05           9  bus, handbag, person, skateboard, stop sign          0.881
     0.10           6  bus, person, stop sign                               0.881
     0.25           5  bus, person                                        0.881
     0.80           4  bus, person                                        0.881
     0.90           0  -                                                  0.000
```

#### 気づいてほしいこと
しきい値を 0.90 にすると、**バスの検出（confidence 0.881）も消えます**。
0.881 < 0.90 だからです。

**「しきい値を上げる＝厳しくする」は、正しいものまで捨てる** ことでもあります。

#### What you should notice
At a threshold of 0.90 the **bus itself disappears**, because its confidence (0.881)
is below 0.90.

Raising a threshold is not simply "being stricter": it also discards correct detections.

---

#### 検出を残す・捨てる境目を見る / Watch where detections are kept or dropped

**推論は 1 回だけ** 行い、その結果をしきい値でふるい分けます。しきい値を動かしても再推論しません。

Inference runs **once**; the threshold only filters that cached result. Moving the threshold does not
re-run the model.

> 推論した最小 confidence は **0.05** です。0.05 より低い検出は存在しないので、
> それより低いしきい値は選べません。
>
> The lowest confidence that was ever inferred is **0.05**. Detections below it do not exist, so no
> lower threshold is offered.

| 予想 / Predict | 見るもの / What to look for |
|---|---|
| しきい値を上げる / raise the threshold | 残る検出の数は？ / how many detections remain? |
| 0.9 にする / set 0.9 | 何も残らない？ / does anything remain? |
| 残った数が 0 / zero remaining | 「異常なし」と同じ意味？ / does that mean "no defect"? |

ウィジェットが無くても、下のパラメータセルで同じ実験ができます。
The parameter cell below runs the same experiment without a widget.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# いちばん低い confidence で 1 回だけ推論し、結果をためます（再学習・再推論はしません）。
# Infer once at the lowest confidence and cache the result; never re-run on slider movement.
MIN_INFERENCE_CONF = 0.05

raw_result = model.predict(source=str(sample_image), conf=MIN_INFERENCE_CONF, imgsz=320,
                           verbose=False)[0]
cached_detections = [
    {
        "class_name": raw_result.names[int(class_id)],
        "conf": float(conf),
        "xyxy": [float(v) for v in box],
    }
    for class_id, conf, box in zip(raw_result.boxes.cls.tolist(),
                                   raw_result.boxes.conf.tolist(),
                                   raw_result.boxes.xyxy.tolist())
]
original_rgb = cv2.cvtColor(raw_result.orig_img, cv2.COLOR_BGR2RGB)

print("推論した最小 confidence / minimum inference confidence:", MIN_INFERENCE_CONF)
print("そのとき残った検出 / detections kept at that confidence:", len(cached_detections))
print("検出が 0 件のときは「異常なし」ではなく「このモデルでは見つからない」です。")
print("Zero detections means \"this model found nothing\", not \"no defect exists\".")

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
import matplotlib.pyplot as plt

yolo_conf_to_try = 0.25      # ← 0.05〜0.95 / 0.05-0.95（推論した最小値より下は不可）

kept = [d for d in cached_detections if d["conf"] >= yolo_conf_to_try]
print(f"しきい値 / threshold = {yolo_conf_to_try:.2f}")
print(f"残った検出 / detections kept = {len(kept)} / {len(cached_detections)}")
for detection in kept:
    print(f"  {detection['class_name']:<12} conf={detection['conf']:.3f}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(original_rgb)
for detection in kept:
    x1, y1, x2, y2 = detection["xyxy"]
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="#a40e26",
                               linewidth=2))
    ax.text(x1, y1 - 4, f"{detection['class_name']} {detection['conf']:.2f}", fontsize=7,
            color="#a40e26")
ax.set_title(f"conf >= {yolo_conf_to_try:.2f} -> {len(kept)} 個 / detections", fontsize=9)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if not aivision.widgets_available():
    aivision.explain_widget_requirement()
else:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    from IPython.display import clear_output, display

    output = widgets.Output()

    def draw(conf_threshold):
        kept = [d for d in cached_detections if d["conf"] >= conf_threshold]
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.imshow(original_rgb)
        for detection in kept:
            x1, y1, x2, y2 = detection["xyxy"]
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor="#a40e26", linewidth=2))
        ax.set_title(f"conf >= {conf_threshold:.2f} -> {len(kept)} 個 / detections", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        with output:
            clear_output(wait=True)
            plt.show()
            plt.close(fig)

    widgets.interact(
        draw,
        conf_threshold=widgets.FloatSlider(value=0.25, min=MIN_INFERENCE_CONF, max=0.95,
                                           step=0.05, description="conf",
                                           continuous_update=False),
    )
    display(output)

## Part 2　座標を読む / Reading the coordinates

#### 座標から何が分かるか
座標を読むと、次のことが分かります。

- 欠陥の **位置**（画像のどこか）
- 欠陥の **大きさ**（幅・高さ）
- 画像の端に接しているか（**途中で切れている** 可能性）
- 同じ位置に複数の検出があるか（**重複検出**）

#### What coordinates tell you
Coordinates tell you:

- **where** the object is
- **how large** it is (width and height)
- whether it touches the border (possibly **cut off**)
- whether several detections sit at the same place (**duplicates**)

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
prediction = model.predict(source=str(sample_image), conf=0.25, verbose=False)[0]
image_height, image_width = prediction.orig_shape

print("画像サイズ / image size (height, width):", prediction.orig_shape)
print()
print(f"{'class':<8} {'w':>7} {'h':>7} {'area':>9} {'w%':>6} {'h%':>6} {'touch border':>14}")
print("-" * 66)
for box in prediction.boxes:
    x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
    name = prediction.names[int(box.cls.item())]
    width = x2 - x1
    height = y2 - y1
    touches = (x1 <= 1) or (y1 <= 1) or (x2 >= image_width - 1) or (y2 >= image_height - 1)
    print(f"{name:<8} {width:>7.1f} {height:>7.1f} {width * height:>9.0f} "
          f"{width / image_width * 100:>5.1f}% {height / image_height * 100:>5.1f}% "
          f"{'YES' if touches else 'no':>14}")

#### 大きさを比で表す
実寸（ピクセル）で比べると、画像サイズが変わったときに意味が分かりにくくなります。
**画像に対する割合（%）** で見ると、画像サイズが変わっても比較できます。

#### Express size as a ratio
Absolute pixels are hard to compare when the image size changes. Expressing the size as
a **percentage of the image** makes comparisons stable.

---

## Part 3　画像を変えて比べる / Comparing different images

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
second_image = Path(ultralytics.__file__).parent / "assets" / "zidane.jpg"

comparison = []
for label, path in [("bus.jpg", sample_image), ("zidane.jpg", second_image)]:
    if not path.is_file():
        print("画像がありません / missing:", path)
        continue
    r = model.predict(source=str(path), conf=0.25, verbose=False)[0]
    names = [r.names[int(c)] for c in r.boxes.cls.tolist()]
    comparison.append({
        "image": label,
        "detections": len(names),
        "unique_classes": ", ".join(sorted(set(names))) if names else "-",
        "mean_conf": round(float(r.boxes.conf.mean().item()), 3) if names else 0.0,
    })
    show([cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)], [f"{label}: {len(names)} detections"], figsize=(7, 5))

print(pd.DataFrame(comparison).to_string(index=False))

---

## Part 4　偽陽性と偽陰性 / False positives and false negatives

#### 4 つの場合
検出結果は、必ず次の 4 つのどれかになります。

| | 実際に欠陥がある | 実際は正常 |
|---|---|---|
| **検出した** | 正しい検出（true positive） | **偽陽性**（false positive, 誤検出） |
| **検出しない** | **偽陰性**（false negative, 見逃し） | 正しい（true negative） |

**製造検査では、偽陰性（見逃し）が特に深刻** です。不良品を出荷してしまうからです。
一方、偽陽性が多いと、ラインが止まりすぎて現場が疲弊します。

ですから **どちらをどれだけ許容するか** を、事前に決めておく必要があります。

#### The four cases
Every detection falls into one of four cases.

| | defect present | actually good |
|---|---|---|
| **detected** | true positive | **false positive** |
| **not detected** | **false negative** | true negative |

In manufacturing, **false negatives are the serious ones** — a defective product ships.
Too many false positives, however, stop the line constantly and exhaust the operators.

You must therefore decide **in advance how much of each you can tolerate**.

#### confidence と正しさは別物
次の 2 つは **どちらも起こります**。

- **低い confidence の検出が正しい** — 小さくて見えにくいが、実在するもの
- **高い confidence の検出が間違っている** — モデルが見た目に騙されている

だから「confidence が高いから大丈夫」とは言えません。
**必ず画像を見て確認する** ことが、この研修でも実務でも基本です。

#### Confidence and correctness are different
Both of these happen:

- **a low-confidence detection can be correct** — small and faint, but real
- **a high-confidence detection can be wrong** — the model is fooled by appearance

So "confidence is high, therefore it is fine" is not a valid argument.
**Always check the image.**

#### 失敗例を記録する
下のセルは、`bus.jpg` の結果から **confidence の低い検出** を
一覧にします。低いものほど「疑わしい」候補です。

演習では、この中から 1 件選んで、
**なぜ低いのか**（小さすぎる、隠れている、見た目が似ている）を考えます。

#### Record a failure case
The next cell lists the **lowest-confidence detections** from `bus.jpg`.
Lower values are the most questionable candidates.

In the exercise you will pick one and explain **why** it is low: too small, occluded, or visually
similar to the class.

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
low_conf = model.predict(source=str(sample_image), conf=0.05, verbose=False)[0]
rows = []
for box in low_conf.boxes:
    x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
    rows.append({
        "class": low_conf.names[int(box.cls.item())],
        "confidence": round(float(box.conf.item()), 3),
        "width_px": round(x2 - x1, 1),
        "height_px": round(y2 - y1, 1),
    })

low_table = pd.DataFrame(rows).sort_values("confidence").reset_index(drop=True)
print(low_table.to_string(index=False))
print()
print("最小の confidence / lowest confidence:", low_table.iloc[0]["confidence"] if len(low_table) else "-")

---

## Part 5　解像度と小さな物体 / Resolution and small objects

### 解像度が欠陥の見え方を決める / resolution decides how a defect looks

![](../../assets/diagrams/teach_resolution_effect.png)

図：同じ合成画像を 640→320→160→80 px に縮小し、欠陥の位置を等倍で表示したもの。Δ は「欠陥あり」と「欠陥なし」の中心画素の差（計算値）。

Figure: one synthetic image shrunk 640 → 320 → 160 → 80 px, zoomed at the defect. Δ is the computed difference at the centre pixel with and without the defect.

**気づいてほしいこと**

画素数が減ると、欠陥の直径は 4.0 → 2.0 → 1.0 → 0.5 px になり、明るさの差も 45 → 38 → 17 → 7 と小さくなります。**撮影する時点で解像度を決めてしまうと、あとからモデルを変えても戻せません。**

**What to notice**

As pixels are removed the defect goes 4.0 → 2.0 → 1.0 → 0.5 px and the difference falls 45 → 38 → 17 → 7. **Resolution is decided when you take the picture; no later model change brings it back.**

**考えてみてください / Try to answer**

- 0.2 mm の欠陥を見つけるには、1 mm あたり何画素必要ですか。この図の 4 つの条件のうち、どれが条件を満たしますか。
- To find a 0.2 mm defect, how many pixels per millimetre do you need, and which of the four conditions here satisfies it?


#### 画像を小さくすると検出できなくなる
モデルは入力を一定の大きさに変換して処理します。
元の画像が小さいと、**小さな物体は数ピクセルしか残りません**。

第 1 日で見たとおり、物体が数ピクセルを下回ると、もう区別できません。

次のセルは、画像を意図的に縮小してから推論し、検出数がどう変わるかを見ます。

#### Shrinking an image loses detections
Models resize their input to a fixed size. If the source image is small, **small
objects survive as only a few pixels**, and as you saw on Day 1, once an object is smaller than a
few pixels it cannot be distinguished.

The next cell downsizes the image before inference and measures what happens to the detection count.

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
source = cv2.imread(str(sample_image))
base_height, base_width = source.shape[:2]

scale_rows = []
work_dir = COURSE_ROOT / "submissions" / "day03"
work_dir.mkdir(parents=True, exist_ok=True)

for scale in [1.0, 0.5, 0.25, 0.15, 0.10]:
    new_size = (int(base_width * scale), int(base_height * scale))
    resized = cv2.resize(source, new_size, interpolation=cv2.INTER_AREA)
    temp_path = work_dir / f"scaled_{int(scale * 100)}.png"
    cv2.imwrite(str(temp_path), resized)

    r = model.predict(source=str(temp_path), conf=0.25, verbose=False)[0]
    scale_rows.append({
        "scale": scale,
        "image_width": new_size[0],
        "image_height": new_size[1],
        "detections": len(r.boxes),
        "classes": ", ".join(sorted({r.names[int(c)] for c in r.boxes.cls.tolist()})) or "-",
    })

print(pd.DataFrame(scale_rows).to_string(index=False))

#### なぜ 0 件になるのか
縮小すると、人の横幅が数十ピクセルから数ピクセルになります。
モデルは「人の形」を認識できなくなります。

これは PVC の 0.3 mm 欠陥で、**さらに深刻** になります。
欠陥が 1 画素未満なら、AI の性能以前の問題です。

#### Why the count drops to zero
After downscaling, a person only a few pixels wide no longer looks like a person to
the model.

For a 0.3 mm PVC defect this is **far worse**: if the defect is smaller than one pixel, no AI can
help. That is an optics problem, not a model problem.

---

## 演習 / Exercises

#### 演習 1（基本）：しきい値の表を作る
Part 1 の表を、`zidane.jpg` でも作り直してください。
しきい値 0.05〜0.90 で、検出数の変化を記録してください。

#### Exercise 1 (basic): build the threshold table
Recreate the Part 1 table for `zidane.jpg` over thresholds 0.05-0.90 and record how the
detection count changes.
Done when the table has 7 rows and you note at which threshold detections drop to zero.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
rows = []
for threshold in [0.05, 0.10, 0.25, 0.40, 0.60, 0.80, 0.90]:
    # r = model.predict(...)
    rows.append({"threshold": threshold, "detections": 0})
print(pd.DataFrame(rows).to_string(index=False))

#### 演習 2（中級）：失敗例を 1 件記録する
上の「低い confidence の一覧」から 1 件選び、記録してください。

| 項目 | 記入 |
|---|---|
| クラス | |
| confidence | |
| 大きさ（幅×高さ） | |
| なぜ低いと思うか | |
| 本当に物体か（画像を見た判断） | |

#### Exercise 2 (intermediate): record one failure case
Pick one entry from the low-confidence table and fill in the record above.
Done when all five fields are filled using your own observation of the image.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
record = {
    "class": "",
    "confidence": 0.0,
    "size_px": "",
    "why_low": "",
    "is_real_object": "",
}
for key, value in record.items():
    print(f"{key:<16} {value if value not in ('', 0.0) else '(未記入 / not written)'}")

#### 演習 3（発展）（発展・任意）：PVC 検査に必要な解像度を提案する
幅 1,300 mm のシートで 0.3 mm の欠陥を、**最低 3 画素** で捉えたいとします。

1. 横方向に必要な画素数を計算してください。
2. その画像を YOLO にそのまま入力できますか。できない場合、どう工夫しますか。
3. Part 5 の結果を根拠に、「画像を縮小して推論する」ことの問題点を説明してください。

#### Exercise 3 (challenge) (challenge, optional) propose the resolution needed for PVC
To capture a 0.3 mm defect on a 1,300 mm sheet with **at least 3 pixels**:

1. compute the required horizontal pixel count
2. can that image be fed to YOLO directly? If not, what would you do?
3. using the Part 5 results as evidence, explain the problem with "shrink the image before inference"
Done when the pixel count is stated and the shrinking problem is explained with evidence.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
sheet_width_mm = 1300.0
defect_mm = 0.3
min_pixels = 3

required_px = 0.0     # <- 計算 / compute
print("1. 必要な横方向の画素数 / required pixels:", required_px)
print("2. そのまま入力できるか / usable directly? :", "")
print("3. 縮小の問題点 / problem with shrinking :", "")

---

#### 理解度チェック / Understanding questions

1. しきい値を 0.9 にすると、なぜ正しい検出まで消えるのですか。
2. 偽陽性と偽陰性を、製造現場の言葉で説明してください。
3. 「confidence が高いから正しい」と言えないのはなぜですか。
4. 画像を縮小すると検出数が減る理由を、2 つ挙げてください。
5. PVC 検査で「見逃し」と「誤報」のどちらを優先して減らすべきですか。あなたの意見と理由を書いてください。

1. Why do correct detections disappear at a threshold of 0.9?
2. Explain false positives and false negatives in shop-floor language.
3. Why can you not say "high confidence means correct"?
4. Give two reasons why shrinking an image reduces detections.
5. In PVC inspection, which should you reduce first: misses or false alarms? Give your view and reasons.
Model answers: `docs/instructor/day03_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| 表が空になる | The table is empty | `conf` が高すぎる可能性があります。0.05 まで下げて確認してください。 | `conf` may be too high; lower it to 0.05. |
| `model` が未定義 | `model` is not defined | Part 0 のモデル読み込みセルを実行してください。 | Run the model-loading cell in Part 0. |
| 縮小した画像でエラーが出る | An error occurs with the resized images | `int()` で整数にしていますが、極端に小さいと 0 になります。`scale` を 0.10 未満にしないでください。 | Very small sizes can become 0; do not go below `scale=0.10`. |
| CSV を Excel で開くと文字化けする | The CSV is garbled in Excel | `encoding="utf-8-sig"` を付けて保存してください。 | Save with `encoding="utf-8-sig"`. |
| 毎回ダウンロードが始まる | It downloads every time | `assets/models/` にファイルがあるか確認してください。無ければ手動で置いてください。 | Check that `assets/models/` contains the file; if not, place it there manually. |

## 完了チェックリスト / Completion checklist

**必須の項目だけ** を並べています。**演習 3（発展）は任意** で、未実施でも完了です。時間が余ったときだけ取り組んでください。

Only the **required** items are listed. **Exercise 3 (challenge) is optional**; leaving it undone still counts as complete. Do it only if you have spare time.

すべてにチェックが付いたら、このノートブックは完了です。

This notebook is complete when every box is checked.

- [ ] Part 1〜Part 5 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-5 and ran every code cell
- [ ] しきい値ごとの検出数の表を作った
      - [ ] Built the threshold-versus-detections table
- [ ] 座標から幅・高さ・割合を求めた
      - [ ] Computed width, height and ratio from coordinates
- [ ] 解像度を下げたときの検出数の変化を確認した
      - [ ] Observed the effect of downscaling
- [ ] 演習 1〜2（必須）を実施した
      - [ ] Completed exercises 1-2 (required)
- [ ] 失敗例を 1 件以上記録した
      - [ ] Recorded at least one failure case
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Ultralytics Predict モード](https://docs.ultralytics.com/modes/predict/)
- [Ultralytics 検証モード](https://docs.ultralytics.com/modes/val/)
- [COCO 評価指標の考え方](https://cocodataset.org/#detection-eval)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

`iou` のしきい値（`model.predict(..., iou=...)`）を変えると、重複検出がどう変わるか調べてください。

Change the NMS `iou` threshold in `model.predict(..., iou=...)` and observe how duplicate detections change.